# Phase 6 — Architecture 3: Legal Prefix–Conv LM

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','-q','install','bert-score','rouge-score'])
from pathlib import Path
import os, json, math, random, sys, pickle, subprocess, zipfile, time
from collections import Counter, defaultdict
import cloudpickle, torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download

# Architecture 3: Legal Prefix–Conv LM — RoPE, RMSNorm, SwiGLU, gated causal convolution
ARCH = 3
BASE = Path('/kaggle/input/datasets/abhishekmahti/a3-p1')
WEIGHTS, BPE = BASE / 'best.pt', BASE / 'tokenizer.json'
OUT = Path('/kaggle/working/phase6_a3_final_1536')
OUT.mkdir(exist_ok=True)
METHODS = ['lora', 'pfeiffer', 'houlsby', 'distill']
SOURCE, TARGET, BATCH = 1536, 640, 1
EPOCHS, PATIENCE, EVAL_STEPS = 8, 7, 250
SEED = 781
tok = Tokenizer.from_file(str(BPE))
PAD = tok.token_to_id('<pad>')
BOS = tok.token_to_id('<bos>')
EOS = tok.token_to_id('<eos>')
VOCAB = tok.get_vocab_size()

In [2]:
def read_rows(split):
    path = hf_hub_download('Exploration-Lab/CS-781-Capstone',
        f'Phase_6_Summ/{split.title()}/{split}.json', repo_type='dataset',
        token=os.environ.get('HF_TOKEN'))
    rows = json.loads(Path(path).read_text())
    if isinstance(rows, dict):
        rows = [dict(v, id=v.get('id', k)) if isinstance(v, dict) else {'id': k, 'document': v}
                for k, v in rows.items()]
    return rows

def prepare(split):
    data = []
    for row in read_rows(split):
        doc = tok.encode(row['document'], add_special_tokens=False).ids
        budget = SOURCE - 1
        if len(doc) > budget:
            quarter = budget // 4
            starts = [0, (len(doc)-quarter)//3, 2*(len(doc)-quarter)//3]
            doc = sum((doc[i:i+quarter] for i in starts), []) + doc[-(budget-3*quarter):]
        target = []
        if split == 'train':
            target = tok.encode(row['summary'], add_special_tokens=False).ids
            if len(target) < TARGET:
                target.append(EOS)
            else:
                target = target[:TARGET]
        data.append((row['id'], doc + [BOS], target))
    return data

rows = prepare('train')
ids = sorted({str(r[0]) for r in rows})
random.Random(SEED).shuffle(ids)
held = set(ids[:max(1, round(.1*len(ids)))])
train = [r for r in rows if str(r[0]) not in held]
dev = [r for r in rows if str(r[0]) in held]
test = prepare('test')
dev_sample = random.Random(SEED).sample(dev, min(32,len(dev)))
wanted = {str(r[0]) for r in dev_sample}
refs = {str(r['id']):r['summary'] for r in read_rows('train') if str(r['id']) in wanted}
with (OUT/'calibration.pkl').open('wb') as f:
    pickle.dump((dev_sample,refs),f)
with (OUT/'data.pkl').open('wb') as f:
    pickle.dump((train,dev,test), f)
print(f'{len(train):,} train | {len(dev):,} dev | {len(test):,} test')

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

6,327 train | 703 dev | 100 test


In [3]:
def norm(x, w):
    y = x.float()
    return (y * torch.rsqrt(y.square().mean(-1, keepdim=True) + 1e-6) * w).to(x.dtype)

def rope(x, cos, sin):
    a, b = x.float().chunk(2, -1)
    return torch.cat((a*cos-b*sin, b*cos+a*sin), -1).to(x.dtype)

class Bottleneck(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.down = nn.Linear(width, 32)
        self.up = nn.Linear(32, width)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x = x + self.up(F.relu(self.down(x)))
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        return x

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d, f = c['width'], c['ff']
        self.h = c['heads']
        self.p = nn.ParameterDict({k: nn.Parameter(torch.empty(*shape)) for k, shape in dict(
            n1=(d,), n2=(d,), nc=(d,), conv=(3,d), gate=(d,),
            qkv=(d,3*d), out=(d,d), up=(d,2*f), down=(f,d)).items()})
        for name, w in self.p.items():
            if name in ('n1','n2','nc'): nn.init.ones_(w)
            elif name == 'gate': nn.init.zeros_(w)
            else: nn.init.normal_(w, std=0.02 / math.sqrt(2*c['layers']) if name in ('out','down') else 0.02)

    def forward(self, x, mask, cos, sin, cache=None, pos=0, capacity=0):
        p = self.p
        u = norm(x, p['nc'])
        history = x.new_zeros(x.size(0),2,x.size(-1)) if cache is None else cache[2]
        z = torch.cat((history,u),1)
        local = sum(z[:,2-i:2-i+x.size(1)] * p['conv'][i].to(x.dtype) for i in range(3))
        x = x + torch.sigmoid(u + p['gate'].to(x.dtype)) * local
        q, k, v = (a.view(x.size(0),x.size(1),self.h,-1).transpose(1,2)
                   for a in (norm(x,p['n1']) @ p['qkv']).chunk(3,-1))
        q, k = rope(q,cos,sin), rope(k,cos,sin)
        if cache is not None:
            cache[0][:,:,pos:pos+x.size(1)].copy_(k)
            cache[1][:,:,pos:pos+x.size(1)].copy_(v)
            cache[2].copy_(z[:,-2:])
            k, v = cache[0][:,:,:pos+x.size(1)], cache[1][:,:,:pos+x.size(1)]
        a = F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.)
        h = a.transpose(1,2).reshape_as(x)
        x = x + h @ p["out"]
        if hasattr(self, "lora_A"): x = x + self.lora_B(self.lora_A(h))
        if hasattr(self, "att_adapter"): x = self.att_adapter(x)
        a, b = (norm(x,p['n2']) @ p['up']).chunk(2,-1)
        x = x + (F.silu(a)*b) @ p['down']
        if hasattr(self, "ff_adapter"): x = self.ff_adapter(x)
        if capacity:
            keys = k.new_zeros(k.size(0),self.h,capacity,k.size(-1)); values = torch.zeros_like(keys)
            keys[:,:,:k.size(2)].copy_(k); values[:,:,:v.size(2)].copy_(v)
            return x, (keys,values,z[:,-2:].clone())
        return x

class Model(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.embed = nn.Parameter(torch.empty(c['vocab'],c['width']))
        nn.init.normal_(self.embed,std=0.02)
        self.final = nn.Parameter(torch.ones(c['width']))
        self.blocks = nn.ModuleList([Block(c) for _ in range(c['layers'])])
        half = c['width']//c['heads']//2
        angle = torch.arange(max(c['prefix']+1024, SOURCE+TARGET)).float()[:,None] * 10000.**(-torch.arange(half).float()/half)
        self.register_buffer('cos', angle.cos()[None,None], persistent=False)
        self.register_buffer('sin', angle.sin()[None,None], persistent=False)

    def hidden(self, ids, prefix):
        t = ids.size(1)
        idx = torch.arange(t,device=ids.device)
        q, k, pre = idx[None,:,None], idx[None,None,:], prefix[:,None,None]
        mask = (((q<pre)&(k<pre)) | ((q>=pre)&(k<=q)))[:,None]
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            args = (x,mask,self.cos[:,:,:t],self.sin[:,:,:t])
            x = checkpoint(block,*args,use_reentrant=False) if self.training else block(*args)
        return norm(x,self.final)

    def forward(self, rows):
        c = self.c
        ids, prefix, length = rows[:,:c['seq']], rows[:,-2], rows[:,-1]
        x = self.hidden(ids,prefix)
        offsets = torch.arange(c['target'],device=rows.device)[None]
        idx = prefix[:,None]-1+offsets
        states = x.gather(1,idx[:,:,None].expand(-1,-1,x.size(-1)))
        labels = ids.gather(1,idx+1)
        logits = states @ self.embed.T
        loss = F.cross_entropy(logits.float().flatten(0,1),labels.flatten(),reduction='none').view_as(labels)
        return ((loss*(offsets<length[:,None])).sum(1)/length).mean()

    def prefill(self, ids):
        t = ids.size(1); cache = []
        x = F.embedding(ids,self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block in self.blocks:
            x, kv = block(x,None,self.cos[:,:,:t],self.sin[:,:,:t],capacity=t+1024)
            cache.append(kv)
        return (norm(x[:,-1],self.final) @ self.embed.T).float(), cache

    def decode(self, token, cache, pos):
        x = F.embedding(token[:,None],self.embed)
        if torch.is_autocast_enabled(): x = x.to(torch.get_autocast_dtype('cuda'))
        for block, kv in zip(self.blocks,cache):
            x = block(x,None,self.cos[:,:,pos:pos+1],self.sin[:,:,pos:pos+1],kv,pos)
        return (norm(x[:,0],self.final) @ self.embed.T).float()

In [4]:
def build(method):
    ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    cfg = dict(ck['config'], gradient_checkpointing=False, pad_id=PAD)
    m = Model(cfg)
    m.load_state_dict(ck['model'])
    m.requires_grad_(False)
    blocks = m.blocks
    width = cfg.get('dim',cfg.get('width'))
    if method in ('houlsby','pfeiffer'):
        for b in blocks:
            b.ff_adapter = Bottleneck(width)
            if method == 'houlsby': b.att_adapter = Bottleneck(width)
    elif method == 'lora':
        for b in blocks:
            b.lora_A = nn.Linear(width,8,bias=False)
            b.lora_B = nn.Linear(8,width,bias=False)
            nn.init.zeros_(b.lora_B.weight)
    elif method == 'distill':
        m.blocks = nn.ModuleList(list(m.blocks)[::2])
        m.requires_grad_(True)
    return m

In [5]:
def logits(m, batch):
    inp = pad([r[1]+r[2][:-1] for r in batch], PAD)
    lens = torch.tensor([len(r[1]) for r in batch], device=inp.device)
    y = pad([[-100]*(len(r[1])-1)+r[2] for r in batch], -100)
    x = m.hidden(inp, lens)
    keep = y != -100
    return F.linear(x[keep], m.embed), y[keep]

In [6]:
def pad(rows, value):
    return nn.utils.rnn.pad_sequence([torch.tensor(r) for r in rows], batch_first=True,
                                     padding_value=value).cuda()

def batches(rows, seed=None):
    order = list(range(len(rows)))
    if seed is not None: random.Random(seed).shuffle(order)
    for start in range(0, len(order), 64):
        chunk = sorted(order[start:start+64], key=lambda i: len(rows[i][1])+len(rows[i][2]))
        for j in range(0,len(chunk),BATCH):
            yield [rows[i] for i in chunk[j:j+BATCH]]

def load_best(method):
    m = build(method)
    m.load_state_dict(torch.load(OUT/method/'best.pt',map_location='cpu',weights_only=True),strict=False)
    return m.cuda().eval()

def train_one(method):
    torch.manual_seed(SEED)
    with (OUT/'data.pkl').open('rb') as f: train, dev, _ = pickle.load(f)
    dest = OUT/method
    dest.mkdir(exist_ok=True)
    m = build(method).cuda()
    params = {k:p for k,p in m.named_parameters() if p.requires_grad}
    teacher = load_best('lora') if method == 'distill' else None
    lr = 2e-5 if method in ('partial','decoder','cross','distill') else 3e-4
    opt = torch.optim.AdamW(params.values(),lr=lr,fused=True)
    scaler = torch.amp.GradScaler('cuda')
    best, stale, step = float('inf'), 0, 0
    start = time.monotonic()
    print(method, 'trainable', sum(p.numel() for p in params.values()), flush=True)
    for epoch in range(EPOCHS):
        for j, batch in enumerate(batches(train, SEED+epoch), 1):
            m.train()
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda',dtype=torch.float16):
                z,y = logits(m,batch)
                loss = F.cross_entropy(z.float(),y)
                if teacher is not None:
                    with torch.no_grad(): t,_ = logits(teacher,batch)
                    kd = F.kl_div((z.float()/2).log_softmax(-1),(t.float()/2).softmax(-1),reduction='batchmean')
                    loss = .5*loss + 2*kd
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params.values(),1.)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step % EVAL_STEPS and j != math.ceil(len(train)/BATCH): continue
            m.eval()
            total, count = torch.zeros((), device='cuda'), 0
            with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16):
                for v in batches(dev):
                    z,y = logits(m,v)
                    total += F.cross_entropy(z.float(),y,reduction='sum')
                    count += len(y)
            score = total.item()/count
            print(method, 'epoch',epoch+1,'step',step,'dev NLL',round(score,4),flush=True)
            if score < best:
                best, stale = score, 0
                torch.save({k:p.detach().half().cpu() for k,p in params.items()},dest/'best.tmp')
                os.replace(dest/'best.tmp',dest/'best.pt')
            else: stale += 1
            if stale >= PATIENCE: break
        if stale >= PATIENCE: break
    (dest/'done').write_text(json.dumps(dict(dev_nll=best,steps=step,seconds=time.monotonic()-start,
                    peak_GiB=torch.cuda.max_memory_allocated()/2**30)))

In [7]:
@torch.inference_mode()
@torch.autocast('cuda',dtype=torch.float16)
def generate(m,doc,limit=TARGET):
    ids = torch.tensor([doc],device='cuda')
    z,cache = m.prefill(ids)
    result, followers = [], defaultdict(set)
    for i in range(limit):
        z = z[:, :VOCAB].float()
        for special in (PAD,BOS):
            if special != EOS: z[:,special] = -torch.inf
        if i < 8: z[:,EOS] = -torch.inf
        if len(result)>=2:
            z[:,list(followers[tuple(result[-2:])])] = -torch.inf
        token = z.argmax(-1)
        n = token.item()
        if n == EOS: break
        if len(result)>=2: followers[tuple(result[-2:])].add(n)
        result.append(n)
        z = m.decode(token,cache,ids.size(1)+i)
    return result

def predict(method):
    m = load_best(method)
    tokenizer = Tokenizer.from_file(str(BPE))
    with (OUT/'data.pkl').open('rb') as f: _,_,test = pickle.load(f)
    from rouge_score.rouge_scorer import RougeScorer
    from bert_score import score as bertscore
    scorer = RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    with (OUT/'calibration.pkl').open('rb') as f: sample,refs = pickle.load(f)
    generated = [generate(m,d) for _,d,_ in sample]
    gold = [refs[str(k)] for k,_,_ in sample]
    caps = [192,384,TARGET]
    texts = [[tokenizer.decode(t[:cap]).strip() for t in generated] for cap in caps]
    _,_,f1 = bertscore(sum(texts,[]),gold*len(caps),lang='en',model_type='roberta-large',
                       batch_size=4,device='cuda',verbose=False)
    metrics = []
    for cap,group,bert in zip(caps,texts,f1.reshape(len(caps),-1).mean(1).tolist()):
        rs = [scorer.score(ref,hyp) for ref,hyp in zip(gold,group)]
        metrics.append(dict(cap=cap,bert=bert,**{k:sum(r[k].fmeasure for r in rs)/len(rs)
                                               for k in ('rouge1','rougeL')}))
    peak = max(r['bert'] for r in metrics)
    chosen = max((r for r in metrics if r['bert'] >= peak-.002),
                 key=lambda r:r['rouge1']+r['rougeL'])
    (OUT/method/'decoding.json').write_text(json.dumps(dict(selected=chosen,results=metrics),indent=2))
    print(method,chosen,flush=True)
    rows = [dict(id=k,summary=tokenizer.decode(generate(m,d,chosen["cap"])).strip()) for k,d,_ in test]
    package(method,rows)

def package(method,rows):
    with zipfile.ZipFile(OUT/f'submission_{method}.zip','w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('prediction.json',json.dumps(rows,ensure_ascii=False))

In [8]:
def run_jobs(names, worker):
    jobs = {}
    queue = list(names)
    payload = OUT/'worker.pkl'
    payload.write_bytes(cloudpickle.dumps(worker))
    command = "import cloudpickle,sys,torch;torch.set_num_threads(1);cloudpickle.load(open(sys.argv[1],'rb'))(sys.argv[2])"
    while queue or jobs:
        for gpu in (0,1):
            if gpu in jobs:
                name,p = jobs[gpu]
                if p.poll() is None: continue
                if p.returncode: raise RuntimeError(f'{name} exited with {p.returncode}')
                del jobs[gpu]
            ready = [n for n in queue if worker != train_one or n != 'distill' or (OUT/'lora'/'done').exists()]
            if ready:
                name = ready[0]
                queue.remove(name)
                print(f'GPU {gpu} | {name}',flush=True)
                jobs[gpu] = name,subprocess.Popen([sys.executable,'-c',command,str(payload),name],
                    env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(gpu)))
        if queue and not jobs: raise RuntimeError('Train lora before distill.')
        if jobs: time.sleep(2)


In [9]:
run_jobs(METHODS, train_one)

GPU 0 | lora
GPU 1 | pfeiffer
lora trainable 294912
pfeiffer trainable 1198656
pfeiffer epoch 1 step 250 dev NLL 3.6174
lora epoch 1 step 250 dev NLL 3.6891
pfeiffer epoch 1 step 500 dev NLL 3.5801
lora epoch 1 step 500 dev NLL 3.6541
pfeiffer epoch 1 step 750 dev NLL 3.5555
lora epoch 1 step 750 dev NLL 3.6327
pfeiffer epoch 1 step 1000 dev NLL 3.5376
lora epoch 1 step 1000 dev NLL 3.6168
pfeiffer epoch 1 step 1250 dev NLL 3.5299
lora epoch 1 step 1250 dev NLL 3.6084
pfeiffer epoch 1 step 1500 dev NLL 3.5215
lora epoch 1 step 1500 dev NLL 3.6014
pfeiffer epoch 1 step 1750 dev NLL 3.5147
lora epoch 1 step 1750 dev NLL 3.5932
pfeiffer epoch 1 step 2000 dev NLL 3.5096
lora epoch 1 step 2000 dev NLL 3.5864
pfeiffer epoch 1 step 2250 dev NLL 3.4933
lora epoch 1 step 2250 dev NLL 3.581
pfeiffer epoch 1 step 2500 dev NLL 3.485
lora epoch 1 step 2500 dev NLL 3.5747
pfeiffer epoch 1 step 2750 dev NLL 3.4817
lora epoch 1 step 2750 dev NLL 3.571
pfeiffer epoch 1 step 3000 dev NLL 3.4749
lora epo

In [10]:
run_jobs(METHODS, predict)

GPU 0 | lora
GPU 1 | pfeiffer


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2342.15it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2404.19it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEX

pfeiffer {'cap': 384, 'bert': 0.8278369307518005, 'rouge1': 0.3945546466187416, 'rougeL': 0.21126785945139556}
lora {'cap': 640, 'bert': 0.8282209634780884, 'rouge1': 0.4494590301990967, 'rougeL': 0.22403979941044325}
GPU 1 | houlsby


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2514.40it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


houlsby {'cap': 640, 'bert': 0.8294627666473389, 'rouge1': 0.40066818521226644, 'rougeL': 0.21345756441657707}
GPU 0 | distill


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 2496.46it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


distill {'cap': 384, 'bert': 0.8344882726669312, 'rouge1': 0.41883518563742833, 'rougeL': 0.21575152730924663}


In [11]:
def overlap(a,b):
    a,b = Counter(a.lower().split()),Counter(b.lower().split())
    return 2*sum((a&b).values())/max(1,sum(a.values())+sum(b.values()))

candidates = defaultdict(list)
for name in METHODS:
    with zipfile.ZipFile(OUT/f'submission_{name}.zip') as z:
        for row in json.loads(z.read('prediction.json')):
            candidates[row['id']].append(row['summary'])
ensemble = []
for key,_,_ in test:
    group = candidates[key]
    scores = [sum(overlap(s,t) for j,t in enumerate(group) if j != i) for i,s in enumerate(group)]
    ensemble.append(dict(id=key, summary=group[max(range(len(group)),key=scores.__getitem__)]))
package('ensemble',ensemble)
print(OUT)

/kaggle/working/phase6_a3_final_1536


## Notes